# VisualAD 批量推理与评估

对 `own_datasets/second/*/test/` 下所有图像进行推理，与 ground truth（路径中的 `ng` / `ok`）对比，
计算准确率、召回率、精确率、F1、ROC AUC 并绘制 ROC 曲线。

Ground truth 规则：
- 路径含 `ng` → 真实标签 = 1（异常）
- 路径含 `ok` → 真实标签 = 0（正常）

推理结果：
- anomaly score ≥ decision_threshold → 预测标签 = 1
- anomaly score < decision_threshold → 预测标签 = 0

**注意**：数据路径在远程服务器上（本地不存在），请确保路径可访问。

In [ ]:
from pathlib import Path
import inspect
import sys
from types import SimpleNamespace

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image, ImageOps
from scipy.ndimage import gaussian_filter
from sklearn.metrics import (
    accuracy_score,
    auc,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.metrics import f1_score, roc_auc_score
from torchvision.transforms import InterpolationMode, Normalize, ToTensor

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (candidate / 'README.md').exists()
        and (candidate / 'VisualAD_lib').exists()
        and (candidate / 'weight').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        'Could not locate the bottle_VisualAD project root from the current working directory.'
    )

# ========== 修改以下路径即可 ==========
# checkpoint 路径
CHECKPOINT_PATH = PROJECT_ROOT / 'runs/datasets_v2_clip_gpu15/epoch_1.pth'

# 测试数据根目录（服务器上）
TEST_DATA_ROOT = PROJECT_ROOT / 'own_datasets/second'

# 推理设备
REQUESTED_DEVICE = 'auto'  # 'auto' / 'cpu' / 'cuda:0'

# 推理参数
SIGMA = 4
DECISION_THRESHOLD = 0.0
PIXEL_THRESHOLD = 0.55
MIN_REGION_AREA = 80

# 是否可视化部分结果
SHOW_SAMPLES = True
# 每类（ok / ng）各展示几张
MAX_SAMPLES_PER_CLASS = 3

assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'
assert TEST_DATA_ROOT.exists(), f'Test data root not found: {TEST_DATA_ROOT}'

print('Project root:', PROJECT_ROOT)
print('Checkpoint:', CHECKPOINT_PATH)
print('Test data root:', TEST_DATA_ROOT)
print('Device:', REQUESTED_DEVICE)
print('Decision threshold:', DECISION_THRESHOLD)


In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import VisualAD_lib
from utils.anomaly_detection import generate_anomaly_map_from_tokens
from utils.feature_transform import create_feature_transform
from utils.scoring import DEFAULT_TOPK_RATIO, reduce_anomaly_map
from VisualAD_lib.constants import OPENAI_DATASET_MEAN, OPENAI_DATASET_STD


def resolve_device(requested_device='auto'):
    if requested_device == 'auto':
        return 'cuda:0' if torch.cuda.is_available() else 'cpu'
    return requested_device


def ensure_real_checkpoint(path: Path):
    prefix = path.read_bytes()[:128]
    if prefix.startswith(b'version https://git-lfs.github.com/spec/v1'):
        raise RuntimeError(
            '当前 checkpoint 仍然是 Git LFS pointer，不是真实权重。请先执行 `git lfs pull`，'
            '或者把真实的 CLIP.pth 放到这个路径。'
        )


def torch_load_compat(path, map_location='cpu'):
    signature = inspect.signature(torch.load)
    if 'weights_only' in signature.parameters:
        return torch.load(path, map_location=map_location, weights_only=False)
    return torch.load(path, map_location=map_location)


def clip_mean_fill_rgb():
    return tuple(int(round(channel * 255)) for channel in OPENAI_DATASET_MEAN)


def pad_short_side_to_square(image: Image.Image, fill_rgb=None):
    if fill_rgb is None:
        fill_rgb = clip_mean_fill_rgb()

    width, height = image.size
    side = max(width, height)
    pad_left = (side - width) // 2
    pad_top = (side - height) // 2
    pad_right = side - width - pad_left
    pad_bottom = side - height - pad_top

    padded = ImageOps.expand(
        image,
        border=(pad_left, pad_top, pad_right, pad_bottom),
        fill=fill_rgb,
    )

    pad_info = {
        'left': pad_left,
        'top': pad_top,
        'right': pad_right,
        'bottom': pad_bottom,
        'square_size': side,
    }
    return padded, pad_info


def prepare_image_like_project(image_path: Path, image_size: int):
    image_pil = Image.open(image_path).convert('RGB')

    padded_pil, pad_info = pad_short_side_to_square(image_pil, fill_rgb=clip_mean_fill_rgb())
    resized_pil = TF.resize(
        padded_pil, [image_size, image_size], interpolation=InterpolationMode.BICUBIC
    )

    tensor = ToTensor()(resized_pil)
    tensor = Normalize(OPENAI_DATASET_MEAN, OPENAI_DATASET_STD)(tensor)

    return {
        'original_pil': image_pil,
        'padded_pil': padded_pil,
        'resized_pil': resized_pil,
        'input_tensor': tensor,
        'pad_info': pad_info,
    }


def load_visualad_bundle(checkpoint_path: Path, requested_device='auto'):
    ensure_real_checkpoint(checkpoint_path)
    device = resolve_device(requested_device)

    checkpoint = torch_load_compat(str(checkpoint_path), map_location=device)
    backbone = checkpoint.get('backbone', 'ViT-L/14@336px')
    image_size = int(checkpoint.get('image_size', 518))
    features_list = checkpoint.get('features_list', [6, 12, 18, 24])
    finetune_info = checkpoint.get('finetune_info', {})

    model, _ = VisualAD_lib.load(backbone, device=device)
    model.eval()
    model.to(device)

    feature_dim = model.visual.embed_dim
    model.visual.anomaly_token.data = checkpoint['anomaly_token'].to(device)
    model.visual.normal_token.data = checkpoint['normal_token'].to(device)

    layer_transforms = nn.ModuleDict()
    if 'layer_transforms' in checkpoint:
        for layer_name, state_dict in checkpoint['layer_transforms'].items():
            hidden_dim = state_dict['mlp.0.weight'].shape[0]
            layer_transforms[layer_name] = create_feature_transform(
                transform_type='mlp',
                input_dim=feature_dim,
                hidden_dim=hidden_dim,
                output_dim=feature_dim,
                dropout=0.0,
            ).to(device)
            layer_transforms[layer_name].load_state_dict(state_dict)
            layer_transforms[layer_name].eval()

    cross_attn = None
    if 'cross_attn' in checkpoint:
        from utils.spatial_cross_attention import build_layer_adaptive_cross_attention

        config = checkpoint.get('cross_attn_config', {})
        cross_attn = build_layer_adaptive_cross_attention(
            layers=features_list,
            embed_dim=feature_dim,
            num_anchors=config.get('num_anchors', 4),
            dropout=config.get('dropout', 0.1),
            res_scale_init=config.get('res_scale_init', 0.01),
        ).to(device)
        cross_attn.load_state_dict(checkpoint['cross_attn'])
        cross_attn.eval()

    return {
        'checkpoint': checkpoint,
        'finetune_info': finetune_info,
        'device': device,
        'model': model,
        'cross_attn': cross_attn,
        'layer_transforms': layer_transforms,
        'backbone': backbone,
        'image_size': image_size,
        'features_list': features_list,
    }


In [ ]:
def normalize_map_for_display(anomaly_map: np.ndarray):
    anomaly_map = anomaly_map.astype(np.float32)
    amap_min = float(anomaly_map.min())
    amap_max = float(anomaly_map.max())
    if amap_max > amap_min:
        return (anomaly_map - amap_min) / (amap_max - amap_min)
    return np.zeros_like(anomaly_map, dtype=np.float32)


def restore_square_map_to_original(square_map: np.ndarray, original_size, pad_info):
    original_width, original_height = original_size
    square_size = int(pad_info['square_size'])

    square_map = square_map.astype(np.float32)
    square_map = cv2.resize(square_map, (square_size, square_size), interpolation=cv2.INTER_LINEAR)

    left = int(pad_info['left'])
    top = int(pad_info['top'])
    cropped_map = square_map[top:top + original_height, left:left + original_width]

    if cropped_map.shape != (original_height, original_width):
        cropped_map = cv2.resize(
            cropped_map, (original_width, original_height), interpolation=cv2.INTER_LINEAR
        )

    return cropped_map


def filter_small_regions(binary_mask: np.ndarray, min_region_area=80):
    binary_mask = binary_mask.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )
    filtered = np.zeros_like(binary_mask)

    for label_idx in range(1, num_labels):
        area = int(stats[label_idx, cv2.CC_STAT_AREA])
        if area >= int(min_region_area):
            filtered[labels == label_idx] = 1

    return filtered.astype(bool)


def build_localization_mask(normalized_map: np.ndarray, pixel_threshold=0.55, min_region_area=80):
    kernel = np.ones((3, 3), dtype=np.uint8)

    binary_mask = (normalized_map >= float(pixel_threshold)).astype(np.uint8)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
    binary_mask = filter_small_regions(binary_mask, min_region_area=min_region_area)

    if binary_mask.any():
        return binary_mask, float(pixel_threshold)

    flat = normalized_map.reshape(-1)
    k = max(1, int(np.ceil(flat.size * 0.01)))
    fallback_threshold = float(np.partition(flat, -k)[-k])
    fallback_mask = (normalized_map >= fallback_threshold).astype(np.uint8)
    fallback_mask = cv2.morphologyEx(fallback_mask, cv2.MORPH_OPEN, kernel)
    fallback_mask = cv2.morphologyEx(fallback_mask, cv2.MORPH_CLOSE, kernel)
    fallback_mask = filter_small_regions(
        fallback_mask, min_region_area=max(16, min_region_area // 2)
    )
    return fallback_mask, fallback_threshold


def make_heatmap_overlay(image_np: np.ndarray, normalized_map: np.ndarray, alpha=0.45):
    heatmap_uint8 = np.clip(normalized_map * 255.0, 0, 255).astype(np.uint8)
    heatmap_bgr = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(heatmap_rgb, alpha, image_np, 1 - alpha, 0)
    return heatmap_rgb, overlay


def infer_single_image(
    image_path: Path,
    bundle,
    sigma=4,
    decision_threshold=0.0,
    pixel_threshold=0.55,
    min_region_area=80,
):
    prepared = prepare_image_like_project(image_path, bundle['image_size'])
    image_tensor = prepared['input_tensor'].unsqueeze(0).to(bundle['device'])

    model = bundle['model']
    cross_attn = bundle['cross_attn']
    layer_transforms = bundle['layer_transforms']
    features_list = bundle['features_list']
    image_size = bundle['image_size']

    with torch.no_grad():
        vision_output = model.encode_image(image_tensor, features_list)
        anomaly_features = vision_output['anomaly_features']
        normal_features = vision_output['normal_features']
        patch_tokens = vision_output['patch_tokens']
        patch_start_idx = vision_output['patch_start_idx']

        patch_features_list = [patch_token[:, patch_start_idx:, :] for patch_token in patch_tokens]
        if cross_attn is not None:
            adapted_list = cross_attn(
                anomaly_features, normal_features, patch_features_list, features_list
            )
            anomaly_features_list = [item['anomaly'] for item in adapted_list]
            normal_features_list = [item['normal'] for item in adapted_list]
        else:
            anomaly_features_list = [anomaly_features] * len(patch_tokens)
            normal_features_list = [normal_features] * len(patch_tokens)

        anomaly_map_list = []
        for layer_idx, patch_feature in enumerate(patch_tokens):
            anomaly_feat_norm = F.normalize(anomaly_features_list[layer_idx], dim=1, eps=1e-8)
            normal_feat_norm = F.normalize(normal_features_list[layer_idx], dim=1, eps=1e-8)

            transform_key = f"layer_{features_list[layer_idx]}"
            if transform_key in layer_transforms:
                batch_size, num_patches, feat_dim = patch_feature.shape
                patch_feature = layer_transforms[transform_key](
                    patch_feature.view(-1, feat_dim)
                ).view(batch_size, num_patches, feat_dim)

            anomaly_map = generate_anomaly_map_from_tokens(
                anomaly_feat_norm,
                normal_feat_norm,
                patch_feature[:, patch_start_idx:, :],
                image_size,
            )
            anomaly_map_list.append(anomaly_map)

        fused_square_map = torch.stack(anomaly_map_list).sum(dim=0).cpu()
        filtered_square_map = gaussian_filter(fused_square_map[0].numpy(), sigma=sigma)
        filtered_square_map_tensor = torch.from_numpy(filtered_square_map).unsqueeze(0)

        score = float(
            reduce_anomaly_map(
                filtered_square_map_tensor,
                mode='topk_mean',
                topk_ratio=DEFAULT_TOPK_RATIO,
            ).item()
        )
        probability = float(torch.sigmoid(torch.tensor(score)).item())
        pred_is_anomaly = score >= float(decision_threshold)

    original_image_np = np.array(prepared['original_pil'])
    original_size_map = restore_square_map_to_original(
        filtered_square_map,
        prepared['original_pil'].size,
        prepared['pad_info'],
    )
    normalized_original_map = normalize_map_for_display(original_size_map)
    heatmap_rgb, heatmap_overlay = make_heatmap_overlay(
        original_image_np, normalized_original_map
    )

    if pred_is_anomaly:
        binary_mask, effective_pixel_threshold = build_localization_mask(
            normalized_original_map,
            pixel_threshold=pixel_threshold,
            min_region_area=min_region_area,
        )
        localized_image, contours = draw_localization_on_image(original_image_np, binary_mask)
    else:
        binary_mask = np.zeros_like(normalized_original_map, dtype=bool)
        effective_pixel_threshold = float(pixel_threshold)
        localized_image = original_image_np.copy()
        contours = []

    return {
        'prediction_label': 'anomaly' if pred_is_anomaly else 'normal',
        'pred_is_anomaly': pred_is_anomaly,
        'score': score,
        'probability': probability,
        'decision_threshold': float(decision_threshold),
        'original_pil': prepared['original_pil'],
        'heatmap_overlay': heatmap_overlay,
        'pad_info': prepared['pad_info'],
    }


In [ ]:
# 该函数已在上面的 cell 中定义（infer_single_image），保留此处是为了让 notebook 独立运行
# 如 notebook 分段执行，确保上面的 cell 先运行

def draw_localization_on_image(image_np: np.ndarray, binary_mask: np.ndarray):
    vis_image = image_np.copy()
    mask_uint8 = (binary_mask.astype(np.uint8) * 255)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        fill_layer = np.zeros_like(vis_image)
        cv2.drawContours(fill_layer, contours, -1, (255, 0, 0), thickness=cv2.FILLED)
        vis_image = cv2.addWeighted(fill_layer, 0.22, vis_image, 0.78, 0)
        cv2.drawContours(vis_image, contours, -1, (255, 0, 0), thickness=2)

    return vis_image, contours


In [ ]:
def collect_test_images(test_root: Path):
    """
    收集所有测试图像路径，并根据路径提取 ground truth 标签。
    
    目录结构: 
      test_root/
        spoon/
          test/
            ng/
              xxx.jpg  -> label=1 (anomaly)
            ok/
              yyy.jpg  -> label=0 (normal)
        其他种类...
    
    返回: List[dict]，每个元素包含 {
        'image_path': Path,
        'category': str,          # 如 'spoon'
        'split': str,             # 'test'
        'gt_type': str,           # 'ng' 或 'ok'
        'gt_label': int,          # 1 (ng) 或 0 (ok)
    }]
    """
    records = []
    
    # 遍历 test_root 下的每个子目录（类别名，如 spoon）
    for category_dir in sorted(test_root.iterdir()):
        if not category_dir.is_dir():
            continue
        category = category_dir.name
        
        # 期望路径: category_dir / 'test' / 'ng'/*.jpg 或 'test' / 'ok'/*.jpg
        test_dir = category_dir / 'test'
        if not test_dir.exists():
            print(f'[WARNING] test dir not found: {test_dir}, skipping.')
            continue
        
        for gt_type in ['ng', 'ok']:
            gt_dir = test_dir / gt_type
            if not gt_dir.exists():
                print(f'[WARNING] {gt_dir} not found, skipping.')
                continue
            
            for img_path in sorted(gt_dir.iterdir()):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']:
                    records.append({
                        'image_path': img_path,
                        'category': category,
                        'split': 'test',
                        'gt_type': gt_type,
                        'gt_label': 1 if gt_type == 'ng' else 0,
                    })
    
    return records


print('Collecting test images...')
test_records = collect_test_images(TEST_DATA_ROOT)
print(f'Total test images found: {len(test_records)}')

if test_records:
    df_summary = {}
    for r in test_records:
        key = (r['category'], r['gt_type'])
        df_summary[key] = df_summary.get(key, 0) + 1
    
    print('\nImages per category & type:')
    for (cat, gt), cnt in sorted(df_summary.items()):
        print(f'  {cat} / {gt}: {cnt}')


In [ ]:
bundle = load_visualad_bundle(CHECKPOINT_PATH, requested_device=REQUESTED_DEVICE)
finetune_info = bundle.get('finetune_info', {})
effective_sigma = finetune_info.get('sigma', SIGMA)
effective_decision_threshold = finetune_info.get('decision_threshold', DECISION_THRESHOLD)

print('Model loaded successfully.')
print('Backbone:', bundle['backbone'])
print('Image size:', bundle['image_size'])
print('Feature layers:', bundle['features_list'])
print('Effective sigma:', effective_sigma)
print('Effective decision threshold:', effective_decision_threshold)
print('Device:', bundle['device'])


In [ ]:
from tqdm import tqdm


def run_batch_inference(records, bundle, sigma, decision_threshold, pixel_threshold, min_region_area):
    """
    对所有记录执行批量推理，返回结果列表。
    
    每条记录新增字段:
        'score': float, anomaly score
        'pred_label': int, 预测标签 (1=异常, 0=正常)
        'probability': float, sigmoid(score)
        'inference_ok': bool, 推理是否成功
        'error': str, 推理失败时的错误信息
    """
    results = []
    for rec in tqdm(records, desc='Inference'):
        item = dict(rec)
        try:
            result = infer_single_image(
                item['image_path'],
                bundle,
                sigma=sigma,
                decision_threshold=decision_threshold,
                pixel_threshold=pixel_threshold,
                min_region_area=min_region_area,
            )
            item['score'] = result['score']
            item['probability'] = result['probability']
            item['pred_label'] = 1 if result['pred_is_anomaly'] else 0
            item['inference_ok'] = True
            item['error'] = None
        except Exception as e:
            item['score'] = None
            item['probability'] = None
            item['pred_label'] = None
            item['inference_ok'] = False
            item['error'] = str(e)
            print(f'\n[ERROR] {item["image_path"]}: {e}')
        results.append(item)
    return results


print('Starting batch inference...')
print(f'Using decision threshold: {effective_decision_threshold}')
print(f'Using sigma: {effective_sigma}')
print(f'Using pixel_threshold: {PIXEL_THRESHOLD}, min_region_area: {MIN_REGION_AREA}')
print()

inference_results = run_batch_inference(
    test_records,
    bundle,
    sigma=effective_sigma,
    decision_threshold=effective_decision_threshold,
    pixel_threshold=PIXEL_THRESHOLD,
    min_region_area=MIN_REGION_AREA,
)

ok_count = sum(1 for r in inference_results if r['inference_ok'])
fail_count = len(inference_results) - ok_count
print(f'\nInference done. Success: {ok_count}, Failed: {fail_count}')


In [ ]:
def compute_metrics(results, decision_threshold):
    """
    基于推理结果计算各项指标。
    
    返回:
        metrics: dict, 包含 accuracy, precision, recall, f1, roc_auc
        df: DataFrame, 每张图的结果
    """
    import pandas as pd
    from sklearn.metrics import confusion_matrix
    from sklearn.metrics import roc_curve, auc
    
    # 过滤掉推理失败的记录
    valid = [r for r in results if r['inference_ok']]
    print(f'Valid samples: {len(valid)} / {len(results)}')
    
    if not valid:
        raise ValueError('No valid inference results!')
    
    y_true = np.array([r['gt_label'] for r in valid])
    y_pred = np.array([r['pred_label'] for r in valid])
    y_scores = np.array([r['score'] for r in valid])
    y_probs = np.array([r['probability'] for r in valid])
    
    # ---- 混淆矩阵 ----
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f'\nConfusion Matrix:')
    print(f'               Predicted Normal  Predicted Anomaly')
    print(f'  Actual Normal      {tn:5d}          {fp:5d}')
    print(f'  Actual Anomaly     {fn:5d}          {tp:5d}')
    
    # ---- 二分类指标 (使用固定阈值) ----
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # ---- ROC AUC ----
    try:
        roc_auc = roc_auc_score(y_true, y_scores)
    except ValueError:
        roc_auc = float('nan')
        print('[WARNING] Cannot compute ROC AUC (only one class present?)')
    
    # ---- ROC 曲线数据 ----
    if roc_auc == roc_auc:  # not NaN
        fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    else:
        fpr, tpr, thresholds = None, None, None
    
    metrics = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': roc_auc,
        'confusion_matrix': cm,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'fpr': fpr,
        'tpr': tpr,
        'roc_thresholds': thresholds,
    }
    
    # ---- DataFrame ----
    df = pd.DataFrame(valid)
    df = df[['image_path', 'category', 'gt_type', 'gt_label', 'score', 'probability', 'pred_label', 'inference_ok']]
    
    return metrics, df


metrics, df_results = compute_metrics(inference_results, effective_decision_threshold)


print('=' * 50)
print('EVALUATION RESULTS (Decision Threshold = {:.4f})'.format(effective_decision_threshold))
print('=' * 50)
print(f'Accuracy:  {metrics["accuracy"]:.4f}')
print(f'Precision: {metrics["precision"]:.4f}')
print(f'Recall:    {metrics["recall"]:.4f}')
print(f'F1 Score:  {metrics["f1"]:.4f}')
print(f'ROC AUC:   {metrics["roc_auc"]:.4f}')
print()
print(f'TP (True Positive / 正确检测异常): {metrics["tp"]}')
print(f'FP (False Positive / 误报):      {metrics["fp"]}')
print(f'TN (True Negative / 正确检测正常): {metrics["tn"]}')
print(f'FN (False Negative / 漏报):      {metrics["fn"]}')


In [ ]:
def plot_roc_curve(metrics, save_path=None):
    fpr = metrics.get('fpr')
    tpr = metrics.get('tpr')
    roc_auc = metrics.get('roc_auc')
    
    if fpr is None or tpr is None:
        print('[WARNING] Cannot plot ROC curve: missing data')
        return
    
    plt.figure(figsize=(8, 6))
    lw = 2
    plt.plot(
        fpr, tpr, color='darkorange', lw=lw,
        label=f'ROC curve (AUC = {roc_auc:.4f})'
    )
    plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--', label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve — VisualAD Bottle-Cap Test Set', fontsize=14)
    plt.legend(loc='lower right', fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'ROC curve saved to: {save_path}')
    
    plt.show()


roc_save_path = PROJECT_ROOT / 'roc_curve.png'
plot_roc_curve(metrics, save_path=roc_save_path)


In [ ]:
import pandas as pd


def compute_per_category_metrics(df, decision_threshold):
    """
    按类别（category）和全体三个维度分别计算指标。
    返回 DataFrame。
    """
    rows = []
    
    # 全体
    all_true = df['gt_label'].values
    all_pred = df['pred_label'].values
    all_scores = df['score'].values
    
    acc = accuracy_score(all_true, all_pred)
    prec = precision_score(all_true, all_pred, zero_division=0)
    rec = recall_score(all_true, all_pred, zero_division=0)
    f1 = f1_score(all_true, all_pred, zero_division=0)
    try:
        auc_val = roc_auc_score(all_true, all_scores)
    except:
        auc_val = float('nan')
    rows.append({
        'Category': 'ALL',
        'Total': len(df),
        'NG': int((df['gt_label'] == 1).sum()),
        'OK': int((df['gt_label'] == 0).sum()),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'ROC_AUC': auc_val,
    })
    
    # 按每个 category 分别统计
    for cat, grp in df.groupby('category'):
        y_true = grp['gt_label'].values
        y_pred = grp['pred_label'].values
        y_scores = grp['score'].values
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc_val = roc_auc_score(y_true, y_scores)
        except:
            auc_val = float('nan')
        rows.append({
            'Category': cat,
            'Total': len(grp),
            'NG': int((grp['gt_label'] == 1).sum()),
            'OK': int((grp['gt_label'] == 0).sum()),
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1': f1,
            'ROC_AUC': auc_val,
        })
    
    metrics_df = pd.DataFrame(rows)
    metrics_df = metrics_df.set_index('Category')
    metrics_df = metrics_df.round(4)
    return metrics_df


df_per_cat = compute_per_category_metrics(df_results, effective_decision_threshold)
print('Per-Category Evaluation Metrics:')
print('(threshold = {:.4f})'.format(effective_decision_threshold))
print()
print(df_per_cat.to_string())

# 保存指标到 CSV
metrics_csv_path = PROJECT_ROOT / 'evaluation_metrics.csv'
df_per_cat.to_csv(metrics_csv_path)
print(f'\nMetrics saved to: {metrics_csv_path}')

# 保存详细结果到 CSV
detail_csv_path = PROJECT_ROOT / 'inference_results.csv'
df_results['image_path'] = df_results['image_path'].apply(lambda p: str(p))
df_results.to_csv(detail_csv_path, index=False)
print(f'Detailed results saved to: {detail_csv_path}')


In [ ]:
# Score 分布直方图（OK vs NG）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ok_scores = df_results[df_results['gt_label'] == 0]['score'].dropna()
ng_scores = df_results[df_results['gt_label'] == 1]['score'].dropna()

axes[0].hist(ok_scores, bins=40, alpha=0.6, label=f'OK (n={len(ok_scores)})', color='green')
axes[0].hist(ng_scores, bins=40, alpha=0.6, label=f'NG (n={len(ng_scores)})', color='red')
axes[0].axvline(effective_decision_threshold, color='black', linestyle='--', linewidth=1.5,
                label=f'Threshold={effective_decision_threshold:.2f}')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Anomaly Score Distribution (OK vs NG)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 各 category 的 score boxplot
categories = sorted(df_results['category'].unique())
data_by_cat = {}
for cat in categories:
    ok_s = df_results[(df_results['category'] == cat) & (df_results['gt_label'] == 0)]['score'].dropna().values
    ng_s = df_results[(df_results['category'] == cat) & (df_results['gt_label'] == 1)]['score'].dropna().values
    data_by_cat[cat] = {'ok': ok_s, 'ng': ng_s}

cat_labels = []
ok_data = []
ng_data = []
for cat in categories:
    cat_labels.append(cat)
    ok_data.append(data_by_cat[cat]['ok'])
    ng_data.append(data_by_cat[cat]['ng'])

positions_ok = np.arange(len(categories)) * 2
positions_ng = positions_ok + 0.6

bp_ok = axes[1].boxplot(ok_data, positions=positions_ok, widths=0.5, patch_artist=True)
bp_ng = axes[1].boxplot(ng_data, positions=positions_ng, widths=0.5, patch_artist=True)

for patch in bp_ok['boxes']:
    patch.set_facecolor('lightgreen')
for patch in bp_ng['boxes']:
    patch.set_facecolor('lightcoral')

axes[1].set_xticks(positions_ok + 0.3)
axes[1].set_xticklabels(cat_labels, rotation=30, ha='right')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Anomaly Score')
axes[1].set_title('Score Distribution by Category (OK=green, NG=red)')
axes[1].axhline(effective_decision_threshold, color='black', linestyle='--', linewidth=1.5)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend([bp_ok['boxes'][0], bp_ng['boxes'][0]], ['OK', 'NG'], loc='upper right')

plt.tight_layout()
score_dist_path = PROJECT_ROOT / 'score_distribution.png'
plt.savefig(score_dist_path, dpi=150, bbox_inches='tight')
print(f'Score distribution saved to: {score_dist_path}')
plt.show()


In [ ]:
# 可视化部分样本
def visualize_samples(df, bundle, max_per_class=3):
    """
    随机展示每类（ok / ng）各 max_per_class 张图的结果。
    展示: 原图 + 热力图叠加 + score + 预测 vs 真实标签。
    """
    import random
    random.seed(42)

    fig_all = []
    
    for gt_type in ['ok', 'ng']:
        subset = df[df['gt_type'] == gt_type].copy()
        subset = subset.sample(n=min(max_per_class, len(subset)), random_state=42)
        
        for _, row in subset.iterrows():
            img_path = Path(row['image_path'])
            score = row['score']
            pred_label = row['pred_label']
            gt_label = row['gt_label']
            cat = row['category']
            
            # 重新推理一次拿到热力图
            try:
                result = infer_single_image(
                    img_path, bundle,
                    sigma=effective_sigma,
                    decision_threshold=effective_decision_threshold,
                    pixel_threshold=PIXEL_THRESHOLD,
                    min_region_area=MIN_REGION_AREA,
                )
                overlay = result['heatmap_overlay']
                
                fig, axes = plt.subplots(1, 2, figsize=(10, 5))
                axes[0].imshow(result['original_pil'])
                axes[0].set_title(f'Original Image')
                axes[0].axis('off')
                
                axes[1].imshow(overlay)
                axes[1].set_title(
                    f'Score: {score:.3f}  |  '
                    f'Pred: {pred_label} ({row["gt_type"]})  |  '
                    f'GT: {gt_label}  |  '
                    f'{cat}'
                )
                axes[1].axis('off')
                
                correct = '✅' if pred_label == gt_label else '❌'
                plt.suptitle(f'{correct} {cat} / {gt_type.upper()}', fontsize=13, y=1.02)
                plt.tight_layout()
                plt.show()
                plt.close()
            except Exception as e:
                print(f'[ERROR] Cannot visualize {img_path}: {e}')


if SHOW_SAMPLES:
    print('Showing sample visualizations...')
    visualize_samples(df_results, bundle, max_per_class=MAX_SAMPLES_PER_CLASS)


In [ ]:
print('=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'Test data root: {TEST_DATA_ROOT}')
print(f'Total images:   {len(df_results)}')
print(f'OK (normal):    {(df_results["gt_label"] == 0).sum()}')
print(f'NG (anomaly):   {(df_results["gt_label"] == 1).sum()}')
print()
print(f'Decision Threshold: {effective_decision_threshold}')
print(f'Sigma:             {effective_sigma}')
print()
print(f'Accuracy:   {metrics["accuracy"]:.4f}')
print(f'Precision:  {metrics["precision"]:.4f}')
print(f'Recall:     {metrics["recall"]:.4f}')
print(f'F1 Score:   {metrics["f1"]:.4f}')
print(f'ROC AUC:    {metrics["roc_auc"]:.4f}')
print()
print('Output files:')
print(f'  ROC curve:      {PROJECT_ROOT / "roc_curve.png"}')
print(f'  Score hist:     {PROJECT_ROOT / "score_distribution.png"}')
print(f'  Metrics CSV:    {PROJECT_ROOT / "evaluation_metrics.csv"}')
print(f'  Results CSV:    {PROJECT_ROOT / "inference_results.csv"}')
print()
print('Per-category results:')
print(df_per_cat.to_string())
